# Real SCLC Patient Example Walkthrough

This notebook walks through the **included real-patient SCLC example** derived from the attached TumorMiner-style files.

It assumes you already have 'sclc_real_patient_panel_dataset.csv' from the real dataset ingestion and analysis dataset creation from steps provided in the README.

This notebook focuses on: cohort overview, real patient metadata fields, expression signature summaries, mutation summaries, optional coarse prior-treatment label exploration, and survival outcome exploration when available in the metadata.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

DATA_PATH = Path("../sclc_real_patient_panel_dataset.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Expected ../sclc_real_patient_panel_dataset.csv. Run the preparation + CLI steps first."
    )

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

In [ ]:
print("Columns:")

for c in df.columns:
    print("-", c)

## Cohort overview

In [ ]:
overview_cols = [
    c
    for c in [
        "sample_id",
        "patient_id",
        "diagnosis",
        "age_at_collection",
        "collection_date",
        "sample_type",
        "biopsy_site",
        "tumor_stage",
        "gender",
        "vital_status",
        "prior_treatment_label",
    ]
    if c in df.columns
]
df[overview_cols].head(10)

In [ ]:
if "sample_type" in df.columns:
    df["sample_type"].value_counts(dropna=False)

In [ ]:
if "biopsy_site" in df.columns:
    df["biopsy_site"].value_counts(dropna=False).head(15)

## Signature summaries

In [ ]:
sig_cols = [c for c in df.columns if c.startswith("SCLC_")]
sig_cols

In [ ]:
if sig_cols:
    df[sig_cols].describe().T

In [ ]:
if sig_cols:
    for col in sig_cols:
        plt.figure(figsize=(6, 4))
        pd.to_numeric(df[col], errors="coerce").hist()
        plt.title(col)
        plt.xlabel("score")
        plt.ylabel("count")
        plt.show()

## Mutation summaries

In [ ]:
mut_cols = [c for c in df.columns if c.startswith("mut_")]
mut_cols

In [ ]:
if mut_cols:
    mut_summary = (
        df[mut_cols]
        .apply(pd.to_numeric, errors="coerce")
        .fillna(0)
        .sum()
        .sort_values(ascending=False)
    )
mut_summary

## Coarse prior-treatment / therapy view

This dataset does **not** contain regimen-level therapy lines from the source bundle. `regimen` here is derived from the metadata's coarse prior-treatment label and should be interpreted cautiously.

In [ ]:
for therapy_col in ["regimen", "prior_treatment_label"]:
    if therapy_col in df.columns:
        print(f"Value counts for {therapy_col}:")
        print(df[therapy_col].value_counts(dropna=False))

In [ ]:
if "regimen" in df.columns and sig_cols:
    group = df.groupby("regimen", dropna=False)[sig_cols].mean(numeric_only=True)
group

## Survival metadata exploration

The included real metadata contains fields like `os_months` and `event_observed` when available. This is metadata exploration only; no synthetic survival augmentation is added in this notebook.

In [ ]:
surv_cols = [c for c in ["os_months", "event_observed", "vital_status"] if c in df.columns]
surv_cols

In [ ]:
if "os_months" in df.columns:
    pd.to_numeric(df["os_months"], errors="coerce").describe()

In [ ]:
if "os_months" in df.columns:
    plt.figure(figsize=(6, 4))
    pd.to_numeric(df["os_months"], errors="coerce").dropna().hist()
    plt.title("Overall survival months (metadata field)")
    plt.xlabel("months")
    plt.ylabel("count")
    plt.show()

## Simple cross-tabs

In [ ]:
if "sample_type" in df.columns and "mut_TP53" in df.columns:
    pd.crosstab(df["sample_type"], df["mut_TP53"], dropna=False)

In [ ]:
if "biopsy_site" in df.columns and sig_cols:
    df.groupby("biopsy_site", dropna=False)[sig_cols].mean(numeric_only=True).head(10)

## Notes

This is a **real-patient example** built from the attached SCLC source files. The expression matrix here is a **panel subset** chosen to align with the toolkit's example signatures and mutation features. The therapy file is **coarse** and derived from `priorTreatment`, not a regimen-level therapy table. Other real data sources can be used as long as they are transformed into the repo's default CSV ingestion format.